In [0]:
import requests
from infos_credenciais import KEY_2

url = 'https://www.datacenters.world/api/v1/facilities'
headers = {f'Authorization': '{KEY_2}'}
params = {
    'offset': 0 
}
list_data = []

while True:
    response = requests.get(url, params=params, headers=headers)
    response.raise_for_status()
    json_data = response.json()
    facilities = json_data.get('data', [])
    for center in facilities:
        list_data.append({
            "id": center.get("slug"),
            "name": center.get("name"),
            "operator": center.get("operator"),
            "region": center.get("region"),
            "iso2": center.get("country")
        })
    meta = json_data.get('meta', {})
    next_offset = meta.get('next_offset')
    
    if not next_offset:
        break
    params['offset'] = next_offset


In [0]:
import pandas as pd

df_data_center = spark.createDataFrame(pd.DataFrame(list_data))

df_data_center.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("portfolio_energia.bronze.datacenter_bronze_raw")
